# Nemotron OCR v2 through vLLM

This notebook demonstrates the local vLLM pooling/plugin wrapper for NVIDIA Nemotron OCR v2. The wrapper does not turn Nemotron OCR into a native vLLM language model; it hosts the OCR pipeline behind vLLM's plugin API and compares the output with the direct Python API.

## Prerequisites

- Python 3.12
- NVIDIA GPU with a CUDA toolkit compatible with your PyTorch build
- `nemotron_ocr` installed from the Hugging Face model repository
- This repository installed with `pip install -e .`

## Why this is a vLLM plugin wrapper

vLLM's Nemotron Parse support is a native generative VLM integration: images become `pixel_values`, a vision tower creates embeddings, and a decoder generates tokens. Nemotron OCR v2 is different: its Hugging Face package exposes an OCR pipeline made of detector, recognizer, relational layout model, and CUDA/C++ post-processing. This notebook therefore uses vLLM's pooling/plugin IO path and verifies direct-vs-vLLM OCR parity instead of claiming native VLM acceleration.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

repo = Path.cwd()
if not (repo / "run_vllm_ocr.py").exists():
    repo = Path.cwd().parent

os.environ.setdefault("HF_HOME", str(Path.home() / ".cache" / "huggingface"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(Path.home() / ".cache" / "huggingface" / "hub"))
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")

repo

In [ ]:
sample = repo / "examples" / "sample_invoice.png"
if not sample.exists():
    subprocess.run([sys.executable, str(repo / "examples" / "make_sample_image.py")], check=True)
sample

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(sample)))

## Run the vLLM wrapper

The command below goes through `vllm.LLM(..., runner="pooling")` and the registered IO processor plugin.

In [ ]:
cmd = [sys.executable, str(repo / "run_vllm_ocr.py"), str(sample)]
completed = subprocess.run(cmd, text=True, capture_output=True, check=True, env=os.environ.copy())
print(completed.stdout)
vllm_output = json.loads(completed.stdout)
vllm_text = " ".join(region["text"] for region in vllm_output["regions"])
vllm_text

## Compare direct OCR and vLLM OCR

The benchmark below runs each backend in a separate subprocess and compares recognized text, region counts, initialization time, loaded latency, and throughput.

In [ ]:
bench_path = repo / "results" / "notebook-benchmark.json"
bench_cmd = [
    sys.executable,
    str(repo / "benchmarks" / "benchmark_ocr.py"),
    "--backend", "both",
    "--images", str(sample),
    "--warmup", "1",
    "--iterations", "3",
    "--output", str(bench_path),
]
subprocess.run(bench_cmd, check=True, env=os.environ.copy())
bench = json.loads(bench_path.read_text())
bench["comparison"]

In [ ]:
summary = {
    "direct_mean_latency_s": bench["direct"]["latency_mean_seconds"],
    "vllm_mean_latency_s": bench["vllm"]["latency_mean_seconds"],
    "direct_text": bench["direct"]["text_signature"],
    "vllm_text": bench["vllm"]["text_signature"],
}
summary